# Clasificación de imágenes histopatológicas de cáncer de mama mediante CNN

**Proyecto académico – Google Colab**

**Dataset:** Breast Histopathology Images (Kaggle)  
**Problema:** clasificación binaria de parches histopatológicos: IDC negativo (0) / IDC positivo (1).

> **Importante:** este notebook está diseñado para generar los resultados reales al ejecutarse. No se incluyen métricas inventadas.


## 1. Introducción

El carcinoma ductal invasivo (IDC) es uno de los tipos más frecuentes de cáncer de mama. El análisis histopatológico permite estudiar características del tejido mediante imágenes microscópicas. En este proyecto se utiliza aprendizaje profundo, específicamente una Red Neuronal Convolucional (CNN), para clasificar pequeños parches de imágenes histopatológicas según la presencia o ausencia de IDC.

El objetivo académico es construir, entrenar y evaluar un modelo de clasificación de imágenes utilizando TensorFlow/Keras en Google Colab.


## 2. Objetivos

### Objetivo general
Desarrollar una CNN capaz de clasificar imágenes histopatológicas en las categorías IDC negativo e IDC positivo.

### Objetivos específicos
- Explorar y comprender la estructura del conjunto de datos.
- Visualizar ejemplos representativos de ambas clases.
- Preparar las imágenes para el entrenamiento.
- Implementar una CNN con TensorFlow/Keras.
- Entrenar y validar el modelo.
- Evaluar su rendimiento mediante accuracy, precision, recall y F1-score.
- Analizar la matriz de confusión y ejemplos de predicción.


## 3. Descripción del dataset

El conjunto Breast Histopathology Images contiene 277.524 parches de 50×50 píxeles provenientes de imágenes histopatológicas de tejido mamario. La clase 0 representa IDC negativo y la clase 1 IDC positivo.

Distribución reportada:
- Clase 0: 198.738 imágenes (aprox. 71,6 %).
- Clase 1: 78.786 imágenes (aprox. 28,4 %).

Esto genera un desequilibrio de clases que debe tenerse en cuenta al interpretar el modelo.


In [ ]:
# 4. Instalación/importación de librerías
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 5. Conexión con Google Drive

Descarga el dataset de Kaggle y colócalo en Google Drive. Ajusta `DATA_DIR` según la carpeta donde lo hayas guardado.

**Alternativa:** puedes descargarlo mediante la API de Kaggle si tienes configuradas tus credenciales.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CAMBIA ESTA RUTA si tu carpeta tiene otro nombre
DATA_DIR = "/content/drive/MyDrive/breast-histopathology-images"

print("Existe la ruta:", os.path.exists(DATA_DIR))


## 6. Exploración de archivos y etiquetas

En este dataset la etiqueta aparece en el nombre del archivo como `class0` o `class1`. La siguiente función extrae la etiqueta automáticamente.


In [ ]:
image_paths = glob.glob(DATA_DIR + "/**/*.png", recursive=True)
print("Imágenes encontradas:", len(image_paths))

def get_label(path):
    name = os.path.basename(path)
    return int(name.split("class")[-1].split(".")[0])

labels = np.array([get_label(p) for p in image_paths])

df = pd.DataFrame({"path": image_paths, "label": labels})
print(df.head())
print("\nDistribución:")
print(df["label"].value_counts())


In [ ]:
# Tabla y porcentaje por clase
distribution = df["label"].value_counts().sort_index()
percentage = distribution / len(df) * 100

summary = pd.DataFrame({
    "Clase": ["IDC negativo (0)", "IDC positivo (1)"],
    "Cantidad": [distribution.get(0,0), distribution.get(1,0)],
    "Porcentaje": [percentage.get(0,0), percentage.get(1,0)]
})

display(summary)


In [ ]:
plt.figure(figsize=(8,5))
ax = sns.countplot(data=df, x="label")
plt.title("Distribución de clases")
plt.xlabel("Clase: 0 = IDC negativo | 1 = IDC positivo")
plt.ylabel("Número de imágenes")

for container in ax.containers:
    ax.bar_label(container)

plt.show()


## 7. Visualización de imágenes

Se muestran ejemplos de ambas clases para comprobar visualmente que el dataset fue cargado correctamente.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13,6))

for row, cls in enumerate([0,1]):
    sample = df[df.label == cls].sample(5, random_state=SEED)
    for ax, (_, item) in zip(axes[row], sample.iterrows()):
        img = Image.open(item["path"])
        ax.imshow(img)
        ax.set_title("IDC negativo" if cls == 0 else "IDC positivo")
        ax.axis("off")

plt.suptitle("Ejemplos de imágenes por clase", fontsize=15)
plt.tight_layout()
plt.show()


## 8. División del conjunto de datos

Se utiliza una división estratificada para conservar aproximadamente la misma proporción de clases:
- 70 % entrenamiento
- 15 % validación
- 15 % prueba


In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print("Entrenamiento:", len(train_df))
print("Validación:", len(val_df))
print("Prueba:", len(test_df))

print("\nProporciones:")
print("Train:", train_df.label.mean())
print("Val:", val_df.label.mean())
print("Test:", test_df.label.mean())


## 9. Preprocesamiento

Las imágenes se redimensionan a 50×50 píxeles y sus valores se normalizan de 0–255 a 0–1.


In [ ]:
IMG_SIZE = (50, 50)
BATCH_SIZE = 64

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.float32)

def make_dataset(dataframe, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["path"].values, dataframe["label"].values)
    )
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(5000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


## 10. Aumento de datos

Se aplican transformaciones moderadas para aumentar la variabilidad de las imágenes durante el entrenamiento y ayudar a reducir el sobreajuste.


In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")


## 11. Arquitectura CNN

La red contiene capas convolucionales para extraer características visuales, Batch Normalization para estabilizar el entrenamiento, MaxPooling para reducir dimensiones, una capa densa para clasificación y Dropout para disminuir el sobreajuste.

La salida utiliza `sigmoid` porque se trata de una clasificación binaria.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(50,50,3)),
    augmentation,

    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model.summary()


## 12. Compilación y entrenamiento

Se utiliza Adam como optimizador y binary cross-entropy como función de pérdida. EarlyStopping detiene el entrenamiento si la validación deja de mejorar.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)


## 13. Curvas de aprendizaje

Estas gráficas permiten comparar el comportamiento de entrenamiento y validación. Una separación creciente entre ambas curvas puede indicar sobreajuste.


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], marker="o", label="Entrenamiento")
plt.plot(history.history["val_accuracy"], marker="o", label="Validación")
plt.title("Accuracy durante el entrenamiento")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], marker="o", label="Entrenamiento")
plt.plot(history.history["val_loss"], marker="o", label="Validación")
plt.title("Loss durante el entrenamiento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()


## 14. Evaluación sobre el conjunto de prueba

In [ ]:
test_results = model.evaluate(test_ds, verbose=1)
print("\nResultados del modelo:")
for name, value in zip(model.metrics_names, test_results):
    print(f"{name}: {value:.4f}")


## 15. Predicciones y reporte de clasificación

In [ ]:
y_true = []
y_prob = []

for images, labels_batch in test_ds:
    probs = model.predict(images, verbose=0).ravel()
    y_prob.extend(probs)
    y_true.extend(labels_batch.numpy())

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(
    y_true, y_pred,
    target_names=["IDC negativo", "IDC positivo"],
    digits=4
))


In [ ]:
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

metricas = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Valor": [accuracy, precision, recall, f1]
})

display(metricas)


## 16. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=["IDC negativo", "IDC positivo"],
    yticklabels=["IDC negativo", "IDC positivo"]
)
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.title("Matriz de confusión")
plt.show()

print("TN:", cm[0,0])
print("FP:", cm[0,1])
print("FN:", cm[1,0])
print("TP:", cm[1,1])


## 17. Ejemplos de predicciones

Se visualizan imágenes del conjunto de prueba junto con su clase real, clase predicha y probabilidad estimada de pertenecer a IDC positivo.


In [ ]:
sample_test = test_df.sample(12, random_state=SEED)

plt.figure(figsize=(13,9))

for i, (_, row) in enumerate(sample_test.iterrows()):
    img = Image.open(row["path"]).convert("RGB")
    arr = np.array(img).astype("float32") / 255.0
    prob = model.predict(arr[None, ...], verbose=0)[0][0]
    pred = int(prob >= 0.5)

    ax = plt.subplot(3,4,i+1)
    ax.imshow(img)
    ax.set_title(
        f"Real: {row['label']} | Pred: {pred}\nP(IDC+)={prob:.2f}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()


## 18. Guardar modelo y métricas

In [ ]:
model.save("/content/modelo_cnn_idc.keras")
metricas.to_csv("/content/metricas_cnn.csv", index=False)

print("Modelo guardado en /content/modelo_cnn_idc.keras")
print("Métricas guardadas en /content/metricas_cnn.csv")


## 19. Conclusiones

La CNN permite construir un flujo completo de clasificación de imágenes histopatológicas. La interpretación final debe basarse en las métricas obtenidas sobre el conjunto de prueba y no solamente en accuracy. Debido al desequilibrio entre clases, recall, precision, F1-score y matriz de confusión son especialmente importantes.

Este proyecto tiene carácter académico y experimental. No debe utilizarse como sistema de diagnóstico médico sin validación clínica, evaluación independiente y controles especializados.


## 20. Referencias

- Kaggle. *Breast Histopathology Images*. Dataset de Paul Mooney.
- LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. *Nature, 521*, 436–444.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
- Chollet, F. (2021). *Deep Learning with Python* (2nd ed.). Manning.
